# Domain 1 · Agentic Architecture & Orchestration (27%)
## Notebook 1 — Task Statement 1.1: Agentic loops

**How to use this notebook.** Read the guide quote, read the plain-English unpacking,
then **run the code cell** and watch the printed `stop_reason`. The abstract bullets
become one observable line. At the end you'll find the exact file:line in your own
course code that does the same thing.

> Setup: this uses the `anthropic` SDK and your `ANTHROPIC_API_KEY`. The cheapest way to
> run it is the same venv you used for the course. From a terminal:
> `cd ../../claude-with-anthropic-api/cli_project && source .venv/bin/activate` then launch
> Jupyter, **or** just `pip install anthropic` in whatever kernel this notebook uses.


---
## 1 · What the guide actually says (verbatim)

> **Task Statement 1.1: Design and implement agentic loops for autonomous task execution**
>
> **Knowledge of:**
> - The agentic loop lifecycle: sending requests to Claude, inspecting `stop_reason`
>   (`"tool_use"` vs `"end_turn"`), executing requested tools, and returning results for
>   the next iteration
> - How tool results are appended to conversation history so the model can reason about
>   the next action
> - The distinction between model-driven decision-making (Claude reasons about which tool
>   to call next based on context) and pre-configured decision trees or tool sequences
>
> **Skills in:**
> - Implementing agentic loop control flow that continues when `stop_reason` is
>   `"tool_use"` and terminates when `stop_reason` is `"end_turn"`
> - Adding tool results to conversation context between iterations so the model can
>   incorporate new information into its reasoning
> - Avoiding anti-patterns such as parsing natural language signals to determine loop
>   termination, setting arbitrary iteration caps as the primary stopping mechanism, or
>   checking for assistant text content as a completion indicator


## 2 · Plain-English unpacking (bullet by bullet)

Think of the loop as a conversation where Claude can pause to ask you to *do* something:

| Guide bullet | In plain words |
|---|---|
| Loop lifecycle / `stop_reason` | Every API response carries a `stop_reason`. `"tool_use"` = *"run this tool and give me the result."* `"end_turn"` = *"I'm done, here's my answer."* You loop while it's `tool_use`. |
| Append tool results to history | After you run the tool, you put its output back into `messages` as a `tool_result` block. The conversation **grows** each turn so Claude sees what happened. |
| Model-driven vs decision tree | **You don't choose the tools.** Claude does, from context. You're not writing `if intent==X: call A`. That hardcoded version is a *script*, not an *agent*. |
| Control flow skill | A `while` loop: continue on `tool_use`, `break` on `end_turn`. |
| Add results between iterations | The single step where you append the `tool_result` before the next API call. Skip it and Claude is blind to the tool's output. |
| Anti-patterns to avoid | ❌ Reading Claude's text for the word "done" to stop. ❌ Using `max_iterations` as the *real* stop. ❌ Treating "there is text" as "it's finished." ✅ The **only** correct stop signal is `stop_reason == "end_turn"`. |


## 3 · Run it — watch `stop_reason` drive the loop

One tiny tool (`get_weather`). Ask about two cities so Claude must call the tool before it
can answer. Watch: turn 0 stops with `tool_use`, you feed results back, turn 1 stops with
`end_turn`. That transition **is** Task Statement 1.1.

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import os, json

# Load the key from your ONE .env (no need to copy it around).
# This notebook lives in ccaf-prep/notebooks/ ; the .env is two levels up
# in claude-with-anthropic-api/. We try that first, then a couple of fallbacks.
_candidates = [
    Path.cwd().parents[1] / "claude-with-anthropic-api" / ".env",   # repo-relative
]
_found = find_dotenv(filename='.env', usecwd=True)   # nearest .env walking up (portable)
_envfile = Path(_found) if _found else next((p for p in _candidates if p.exists()), None)
if _envfile:
    load_dotenv(_envfile)
    print(f"Loaded .env from: {_envfile}")
else:
    print("WARNING: no .env found in", [str(p) for p in _candidates])

assert os.environ.get("ANTHROPIC_API_KEY"), \
    "ANTHROPIC_API_KEY still not set — check the .env path printed above."

client = Anthropic()              # now reads ANTHROPIC_API_KEY from the loaded env
# Cheap by default — these demos teach the MECHANISM, not model IQ, so Haiku is plenty.
# Set CLAUDE_MODEL=claude-sonnet-4-6 in the .env to bump every cell at once.
MODEL = os.environ.get("CLAUDE_MODEL") or "claude-haiku-4-5"
print("Using model:", MODEL)

# --- one tool. Note: WE never decide when to call it; Claude does (model-driven). ---
# These three fields (name, description, input_schema) are the ONLY thing Claude sees.
# It never sees the function body below — it fills `input` from the schema + user text.
TOOLS = [{
    "name": "get_weather",
    "description": "Get the current weather for a city. Returns a short text summary.",
    "input_schema": {"type": "object",
                     "properties": {"city": {"type": "string"}},
                     "required": ["city"]},
}]

def get_weather(city):
    fake = {"Lima": "19C, cloudy", "Tokyo": "24C, sunny"}
    return fake.get(city, "15C, rainy")

# --- name -> function registry: the habit you want (real dispatch, not hardcoding). ---
# Claude returns the tool NAME it chose; we look up the matching function by that name.
# Adding a 2nd tool = add it to TOOLS (so Claude can see it) AND here (so we can run it).
TOOL_FUNCS = {"get_weather": get_weather}

In [ ]:
# pyright: reportUndefinedVariable=false   (TOOLS / client / MODEL / TOOL_FUNCS come from the setup cell above)
def run(user_text, max_turns=10):
    messages = [{"role": "user", "content": user_text}]

    for turn in range(max_turns):          # max_turns is a SAFETY backstop, NOT the stop
        resp = client.messages.create(
            model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)

        print(f"turn {turn}: stop_reason = {resp.stop_reason!r}")   # <-- the heartbeat

        if resp.stop_reason == "end_turn":                          # <-- the ONLY real stop
            answer = "".join(b.text for b in resp.content if b.type == "text")
            print("\nFINAL:", answer)
            return

        if resp.stop_reason == "tool_use":
            # 1) keep Claude's turn (it contains the tool_use request) in history
            messages.append({"role": "assistant", "content": resp.content})
            # 2) run each requested tool and collect results
            results = []
            for block in resp.content:
                if block.type == "tool_use":
                    # DISPATCH BY NAME: look up the function Claude actually named.
                    # (With one tool this is trivial, but it's the correct habit:
                    #  the loop now works for ANY number of tools, unchanged.)
                    func = TOOL_FUNCS.get(block.name)
                    if func is None:
                        out = f"ERROR: unknown tool {block.name!r}"
                        is_error = True
                    else:
                        out = func(**block.input)        # **input unpacks the dict to kwargs
                        is_error = False
                    print(f"   -> Claude called {block.name}({block.input}) = {out!r}")
                    results.append({"type": "tool_result",
                                    "tool_use_id": block.id,
                                    "content": out,
                                    "is_error": is_error})   # tells Claude if the tool failed
            # 3) APPEND results to history so next turn can reason about them
            messages.append({"role": "user", "content": results})
            continue                                                # <-- loop again

run("What's the weather in Lima and Tokyo? Which is warmer?")

### What you just saw → the guide bullets

| Output line | Guide bullet it proves |
|---|---|
| `turn 0: stop_reason = 'tool_use'` | lifecycle: Claude requests a tool |
| `-> Claude called get_weather(...)` | **model-driven** selection (you wrote no routing) |
| `messages.append({... tool_result ...})` | results appended to history between iterations |
| `turn 1: stop_reason = 'end_turn'` | correct termination signal |
| `max_turns` never triggered | it's a backstop, not the stop condition |


### How did Claude know to send `{"city": "Lima"}`? — the `input_schema` contract

You never wrote code that paired `"city"` with `"Lima"`. Claude did. Here's the mechanism,
because it underpins **all** tool use (Domain 2 leans on it heavily):

**The only thing Claude sees about your tool is the three fields you sent:** `name`,
`description`, and `input_schema`. That's the entire contract. The Python function body —
the `fake = {...}` dict, the `.get()` fallback — runs on *your* machine and is **invisible
to Claude.**

The `input_schema` is **JSON Schema** — a fill-in-the-blank form:

```python
"input_schema": {"type": "object",
                 "properties": {"city": {"type": "string"}},
                 "required": ["city"]}     # "to call me, give me a string named city"
```

So the flow is:

1. Claude reads the user text *"weather in Lima and Tokyo?"*
2. Claude reads the tool's `description` → decides `get_weather` is relevant.
3. Claude reads the `input_schema` → sees it must supply a string `city`.
4. Claude **extracts** the cities from the sentence and fills the form: `{"city": "Lima"}`,
   then `{"city": "Tokyo"}`. The SDK parses that JSON into the Python dict you see as
   `block.input`.

**Why only those two cities?** Because they're the only ones *in the question*. Claude
picks inputs from the user's text, **not** from your `fake` dict — which it can't see.

**If `fake` had 100 cities, would you return all 100?** No. Claude made **one call per city
it asked about** (2 calls → 2 results); each returns one string. The size of your dict is
irrelevant. If you *wanted* an "all cities" tool, you'd define a different tool whose schema
and function describe that.

> **Mental model:** the schema is a form Claude fills from context; your function is a back
> office Claude never enters — it slides the filled form under the door and gets a result back.


## 4 · The anti-pattern (so you recognize the distractor)

The exam's wrong answers often *look* reasonable. Here are the three 1.1 anti-patterns as
code. Read them and feel why each is fragile — **do not** use these to stop a loop:


In [ ]:
# ANTI-PATTERN 1: parse natural-language text to decide when to stop.
#   text = response_text(resp)
#   if "done" in text.lower() or "let me know" in text.lower():
#       break        # FRAGILE: phrasing varies; Claude says 'done' mid-task; misses tool_use

# ANTI-PATTERN 2: use the iteration cap as the PRIMARY stop.
#   for _ in range(3):            # stops after 3 no matter what -> truncates real work
#       ...

# ANTI-PATTERN 3: treat 'there is assistant text' as completion.
#   if any(b.type == 'text' for b in resp.content):
#       break        # WRONG: a turn can contain BOTH text and a tool_use request

print("The only correct stop signal is:  resp.stop_reason == 'end_turn'")


## 5 · The same loop in code you already wrote

You are not learning something new — you built this in the course. Compare:

- **Your course code:** `claude-with-anthropic-api/cli_project/core/chat.py`, lines **24–45**.
  The `while True:` with `if response.stop_reason == "tool_use": ... else: break` is exactly
  this notebook's loop, just wrapped in a class.
- **Exercise 1 (fuller version):** `ccaf-prep/exercises/01-support-agent/agent.py` — same loop, plus an
  enforcement gate and structured errors (those belong to Task Statements 1.4 / 2.2).

Open `chat.py:24-45` now and map each line to a bullet above. If you can do that, 1.1 is done.

## 6 · Self-check (cover the answers)

1. The loop should terminate when `stop_reason == ____`.
2. True/False: a single response can contain both assistant text **and** a `tool_use` request.
3. Why is `for _ in range(5)` a bad *primary* stopping mechanism?
4. Who decides which tool gets called next — your code or the model?

<details><summary>answers</summary>

1. `"end_turn"`.  2. True — that's why you must check `stop_reason`, not "is there text".
3. It truncates legitimately long tasks and isn't tied to task completion; it's only a safety
backstop.  4. The model (model-driven), from context — not a hardcoded decision tree.
</details>


## 1.2 · Coordinator–subagent orchestration

**What the guide says (verbatim):**

> **Task Statement 1.2: Orchestrate multi-agent systems with coordinator-subagent patterns**
>
> **Knowledge of:**
> - Hub-and-spoke architecture where a coordinator agent manages all inter-subagent
>   communication, error handling, and information routing
> - How subagents operate with isolated context—they do not inherit the coordinator's
>   conversation history automatically
> - The role of the coordinator in task decomposition, delegation, result aggregation, and
>   deciding which subagents to invoke based on query complexity
> - Risks of overly narrow task decomposition by the coordinator, leading to incomplete
>   coverage of broad research topics
>
> **Skills in:**
> - Designing coordinator agents that analyze query requirements and dynamically select
>   which subagents to invoke rather than always routing through the full pipeline
> - Partitioning research scope across subagents to minimize duplication (e.g., assigning
>   distinct subtopics or source types to each agent)
> - Implementing iterative refinement loops where the coordinator evaluates synthesis output
>   for gaps, re-delegates to search and analysis subagents with targeted queries, and
>   re-invokes synthesis until coverage is sufficient
> - Routing all subagent communication through the coordinator for observability, consistent
>   error handling, and controlled information flow

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| Hub-and-spoke | One **coordinator** in the middle; **subagents** are spokes. Everything flows through the hub — spokes never talk to each other. |
| Isolated context | A subagent starts with a **blank conversation**. It does NOT see the coordinator's history. If it needs a fact, you must put it in *its* prompt. |
| Coordinator role | The coordinator **decomposes** the task, **decides which** subagents to call (based on the query), and **aggregates** their results. |
| Dynamically select | Don't always run the full pipeline. A simple query may need 1 subagent; a broad one needs many. Pick per query. |
| Partition scope | Give each subagent a **distinct** slice (different subtopic / source) so they don't duplicate work. |
| Risk: too narrow | If the coordinator splits the task too finely, broad topics get **incomplete coverage** — a gap, not a wrong answer. |
| Iterative refinement | Coordinator checks the synthesis for **gaps**, re-delegates with targeted queries, re-synthesizes until coverage is enough. |

**Run it — the REAL Agent SDK, hub-and-spoke.** This is not a simulation. `query()` runs a real
coordinator that **delegates** via the `Task` tool. Each subagent is an `AgentDefinition` **scoped to
its own tool** (the SDK enforces it — the weather subagent can't call `get_landmark`), and each gets
**isolated context** automatically. Permissioning is **real** (`permission_mode="default"` + an explicit
`allowed_tools`), not bypassed. Watch the `task_started` spawns and the scoped tools fire. (Runs on
Haiku via your Claude Code auth — ~$0.03–0.04, a few seconds.)

In [ ]:
# 1.2 — REAL coordinator-subagent orchestration with the Agent SDK's Task tool (hub-and-spoke).
# NOT a simulation: query() runs a real coordinator that delegates via the Task tool. Each subagent
# is an AgentDefinition SCOPED to its own tool (the weather subagent literally cannot call
# get_landmark — the SDK enforces it); each subagent also gets ISOLATED context automatically.
# Permissioning is REAL: permission_mode="default" + an explicit allowed_tools (no bypass).
# Cheap by design: Haiku + a turn cap (the lesson is the orchestration, not model IQ).
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, AgentDefinition,
    AssistantMessage, ToolUseBlock, ResultMessage, SystemMessage,
)
SDK_MODEL = "haiku"   # the SDK takes short names ("haiku"/"sonnet"); cheap on purpose

# D2.3: two scoped in-process tools (canned data stands in for real backends). Match on a
# substring so "Tokyo", "Tokyo, Japan", etc. all hit the same canned record.
@tool("get_weather", "Get a city's current weather.", {"city": str})
async def sdk_get_weather(args):
    text = "18C, cloudy" if "tokyo" in args["city"].lower() else "15C, rainy"
    return {"content": [{"type": "text", "text": text}]}

@tool("get_landmark", "Get a city's single most famous landmark.", {"city": str})
async def sdk_get_landmark(args):
    text = "Tokyo Tower (333m)" if "tokyo" in args["city"].lower() else "unknown"
    return {"content": [{"type": "text", "text": text}]}

research_server = create_sdk_mcp_server("research", tools=[sdk_get_weather, sdk_get_landmark])
W, L = "mcp__research__get_weather", "mcp__research__get_landmark"

# D1.3: each subagent is an AgentDefinition — description + system prompt + SCOPED tools.
AGENTS = {
    "weather":  AgentDefinition(description="Researches a city's weather.",
        prompt="Call get_weather for the city, then report it in ONE short sentence.",
        tools=[W], model=SDK_MODEL),
    "landmark": AgentDefinition(description="Researches a city's famous landmark.",
        prompt="Call get_landmark for the city, then report it in ONE short sentence.",
        tools=[L], model=SDK_MODEL),
}

async def run_coordinator(prompt, agents):
    """Run one real coordinator. Returns (spawns, result). Delegates via Task; each subagent is
    scoped to its own tool via AgentDefinition.tools."""
    options = ClaudeAgentOptions(
        model=SDK_MODEL, mcp_servers={"research": research_server}, agents=agents,
        # allowed_tools = the GLOBAL execution allow-list (no bypass): permit "Task" plus the
        # MCP data tools the subagents run, or those tool calls get denied.
        allowed_tools=["Task", W, L],
        permission_mode="default",            # REAL permissioning, not bypassPermissions
        max_turns=16)                          # cost backstop
    spawns, result = [], None
    async for msg in query(prompt=prompt, options=options):
        # A subagent spawn surfaces as SystemMessage(subtype="task_started") — NOT a ToolUseBlock
        # named "Task". Its .data carries subagent_type + the coordinator-authored prompt.
        if isinstance(msg, SystemMessage) and getattr(msg, "subtype", None) == "task_started":
            spawns.append(msg.data)
            print(f"[coordinator -> Task spawns]: {msg.data.get('subagent_type')!r}")
        elif isinstance(msg, AssistantMessage):
            for b in msg.content:
                if isinstance(b, ToolUseBlock) and b.name.startswith("mcp__research__"):
                    print(f"   [scoped subagent tool fired]: {b.name}")
        elif isinstance(msg, ResultMessage):
            result = msg
    return spawns, result

COORD = ("You are a research COORDINATOR. Delegate to the `weather` subagent and the `landmark` "
         "subagent (via the Task tool, one each) to get Tokyo's weather and most famous landmark, "
         "then give ONE two-sentence briefing combining their findings.")

# Top-level await: Jupyter runs this directly (don't wrap in asyncio.run inside the kernel).
spawns, result = await run_coordinator(COORD, AGENTS)
print("\n[FINAL BRIEFING]\n" + (result.result or "(none)"))
names = [s.get("subagent_type") for s in spawns]
if names:
    print(f"\nhub-and-spoke: coordinator spawned {names} "
          "— each got ISOLATED context automatically (it never saw the others).")
else:
    print("\n(no subagents spawned this run — with a live model the coordinator "
          "occasionally answers directly; re-run the cell to see the Task delegation.)")
cost = f"${result.total_cost_usd:.4f}" if result.total_cost_usd else "n/a"
print(f"(real run on {SDK_MODEL}: turns={result.num_turns}, cost={cost})")

**FAQ — when does this actually *run*, and how does permissioning work?** (added from a Q&A)

*Where is the call?* Everything above is just **definitions** — `@tool`, `AGENTS`, `async def run_coordinator` only sit in memory. The engine fires on **one line**: `spawns, result = await run_coordinator(COORD, AGENTS)`. Nothing happens until then.

*Is `ClaudeAgentOptions(...)` a function?* It looks like one, but it's a **constructor** — it just bundles config into a single object (model, subagents, `allowed_tools`, `max_turns`). Nothing executes; `options` is a settings bag you hand off.

*Where is the real work?* Inside `run_coordinator`: `async for msg in query(prompt=..., options=options)`. **`query()` is the SDK's engine** — it starts the coordinator and **streams messages back** (spawns, tool calls, final result); the `async for` loop just *inspects* each message. `await` works at the top level because Jupyter already runs an event loop (a plain `.py` needs `asyncio.run(...)` — see `exercises/04-multi-agent-research/coordinator_sdk.py`).

*Why `permission_mode="default"` and not `bypassPermissions`?* Best practice: use **real permissioning**, not a bypass. `bypassPermissions` would switch off the very allow-list this section teaches (and let *any* tool run). Three facts we measured on this SDK build (Haiku):

| What we tried | Result |
|---|---|
| subagent's MCP tool **not** in `allowed_tools` (default mode) | ❌ **denied** — *"I don't have permission…"* → the data tools **must** be listed |
| MCP tool **in** `allowed_tools` (default mode) | ✅ runs — this is what the cell does |
| `Task` **omitted** from `allowed_tools` (default mode) | ⚠️ coordinator **still spawned** — `Task` is a built-in; "must include `Task`" is the documented contract, not a hard gate here |

**Two layers — don't conflate them:**
- `allowed_tools` = the **global execution allow-list** (one gate for the whole run; there is *no* per-agent permission).
- `AgentDefinition(tools=[…])` = the **per-subagent scope**, and *this* is what's actually enforced: the weather subagent (`tools=[W]`) literally cannot call `get_landmark` (verified). That is what makes hub-and-spoke real — not a permission trick.

So the coordinator delegates because its **prompt** instructs it to, and each subagent is **scoped** to one tool — clean, honest, no bypass.

**The anti-patterns (exam distractors)** — read them and feel why each is wrong:

In [ ]:
# ANTI-PATTERN 1: assume subagents INHERIT the coordinator's context.
#   delegate "continue from what we discussed" -> they can't; the SDK gives each subagent
#   ISOLATED context, so put the needed facts in its prompt (1.3).
# ANTI-PATTERN 2: ALWAYS run the full pipeline regardless of the query.
#   for a in ALL_AGENTS: a.run()    # coordinator should SELECT subagents by complexity.
# ANTI-PATTERN 3: let subagents talk DIRECTLY to each other (bypassing the hub).
#   weather_agent.send(landmark_agent, ...)   # breaks observability + error handling.
# ANTI-PATTERN 4 (sample Q7): blame the wrong component when coverage is incomplete —
#   the usual root cause is the COORDINATOR decomposing too narrowly, not a subagent.
print("Right: coordinator decomposes, SELECTS subagents by query, routes ALL comms, aggregates.")

**In your own code — now built (real SDK).** Exercise 4:
`ccaf-prep/exercises/04-multi-agent-research/coordinator_sdk.py` is the fuller version of the cell above — a
real coordinator that delegates via the `Task` tool to subagents that each own a scoped `@tool`,
with the SDK isolating every subagent's context. Run it
(`cd ccaf-prep/exercises/04-multi-agent-research && uv run python coordinator_sdk.py`) and watch the same
`task_started` spawns and scoped-tool fires you saw here.

**Self-check** (cover the answers)

1. Do subagents automatically see the coordinator's conversation history?
2. When coverage of a broad topic is incomplete, what's the usual root cause?
3. Should the coordinator always invoke every subagent?

<details><summary>answers</summary>

1. **No** — isolated context; you must pass needed facts in the subagent's prompt.
2. The **coordinator decomposed too narrowly** (a gap), not a broken subagent.
3. **No** — it dynamically selects which subagents to invoke based on query complexity.
</details>

## 1.3 · Subagent invocation & context passing

**What the guide says (verbatim):**

> **Task Statement 1.3: Configure subagent invocation, context passing, and spawning**
>
> **Knowledge of:**
> - The Task tool as the mechanism for spawning subagents, and the requirement that
>   `allowedTools` must include `"Task"` for a coordinator to invoke subagents
> - That subagent context must be explicitly provided in the prompt—subagents do not
>   automatically inherit parent context or share memory between invocations
> - The AgentDefinition configuration including descriptions, system prompts, and tool
>   restrictions for each subagent type
> - Fork-based session management for exploring divergent approaches from a shared analysis
>   baseline
>
> **Skills in:**
> - Including complete findings from prior agents directly in the subagent's prompt (e.g.,
>   passing web search results and document analysis outputs to the synthesis subagent)
> - Using structured data formats to separate content from metadata (source URLs, document
>   names, page numbers) when passing context between agents to preserve attribution
> - Spawning parallel subagents by emitting multiple Task tool calls in a single coordinator
>   response rather than across separate turns
> - Designing coordinator prompts that specify research goals and quality criteria rather than
>   step-by-step procedural instructions, to enable subagent adaptability

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| Task tool + `allowedTools` | In the Agent SDK you spawn a subagent by calling the **Task** tool. The coordinator's `allowedTools` must include `"Task"` or it literally can't spawn anyone. |
| Context in the prompt | Because context is isolated (1.2), you **paste the prior findings into the subagent's prompt**. No prompt = no knowledge. |
| Structured data + attribution | Pass findings as **structured** data (JSON) so content stays separate from metadata (which source said what). Preserves citations. |
| Parallel spawning | Emit **multiple Task calls in ONE response** → they run in parallel. Across separate turns = serial = slower. |
| Goals over procedures | Tell subagents the **goal + quality bar**, not step-by-step instructions, so they can adapt. |
| AgentDefinition | Each subagent type has a description, system prompt, and **tool restrictions**. |

**Run it — context passing is REAL here.** Because each subagent has isolated context (1.2), the
coordinator must put what a subagent needs **into its prompt**. We add a `synthesis` subagent with
**no tools** and tell the coordinator to delegate the final write-up to it — *copying both findings
into its prompt*. Then we print the prompt the coordinator actually sent each subagent: you'll see
the synthesis subagent's prompt literally contains the weather + landmark findings. (~$0.04 on Haiku.)

In [ ]:
# 1.3 — CONTEXT PASSING, for real. Reuses run_coordinator/AGENTS/SDK_MODEL from 1.2 (run that
# cell first). Isolated context (1.2) means a subagent knows ONLY what the coordinator puts in its
# prompt. We add a `synthesis` subagent with NO tools and watch the coordinator hand it the prior
# findings IN ITS PROMPT — the exact skill the guide names (findings -> synthesis prompt).
AGENTS3 = {
    **AGENTS,
    "synthesis": AgentDefinition(
        description="Writes the final briefing from findings passed in its prompt. Has NO tools.",
        prompt="You have NO tools and NO memory of prior work. Using ONLY the findings in your "
               "prompt, write ONE briefing sentence, citing each source in [brackets].",
        tools=[], model=SDK_MODEL),   # tool restriction: synthesis gets ZERO data tools
}
COORD3 = (
    "You are a research COORDINATOR for the city Tokyo. Each step via the Task tool:\n"
    "1) Delegate to the `weather` subagent to get Tokyo's weather.\n"
    "2) Delegate to the `landmark` subagent to get Tokyo's most famous landmark.\n"
    "3) Delegate the FINAL write-up to the `synthesis` subagent. It has isolated context, so you "
    "MUST copy BOTH findings verbatim into its prompt. Do NOT write the briefing yourself.\n"
    "Return the synthesis subagent's briefing.")

spawns3, result3 = await run_coordinator(COORD3, AGENTS3)

# PROOF (guide 1.3): print the prompt the coordinator PASSED to each subagent. The synthesis
# subagent's prompt should literally embed the weather + landmark findings -> context passed
# IN THE PROMPT (attributed), NOT inherited.
for s in spawns3:
    print(f"\n--- prompt the coordinator passed to {s.get('subagent_type')!r} ---")
    print((s.get("prompt") or "(none)").strip())
print("\n[FINAL BRIEFING from the synthesis subagent]\n" + (result3.result or "(none)"))

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: expect the subagent to "remember" earlier results -> it can't (isolated).
# ANTI-PATTERN 2: pass context as one flat blob, losing attribution (which source said what).
# ANTI-PATTERN 3: spawn subagents across SEPARATE turns when they could run in PARALLEL:
#   # parallel = multiple Task calls in ONE coordinator response:
#   # content = [ {"type":"tool_use","name":"Task",...},
#   #             {"type":"tool_use","name":"Task",...} ]
# ANTI-PATTERN 4 (config): forget that allowedTools must include "Task" -> coordinator
#   cannot spawn anyone at all.
print("Right: pass COMPLETE findings in the prompt, structured + attributed; parallel = "
      "multiple Task calls in one response; allowedTools must include 'Task'.")

**In your own code — now built (real SDK).** Same `exercises/04-multi-agent-research/coordinator_sdk.py`: the
`AGENTS` map is `AgentDefinition` config in the flesh (description / system prompt / scoped
`tools`), the coordinator's `allowed_tools=["Task"]` is the `allowedTools` requirement, and each
`task_started` spawn carries the coordinator-authored `prompt` — the only context an isolated
subagent ever sees.

### What you just ran, in exam vocabulary

The two cells above are the **real Agent SDK** (Python `claude-agent-sdk`), not a simulation. Map
what you saw to the terms the exam uses:

| Guide bullet | What you ran |
|---|---|
| `allowedTools` must include `"Task"` | `allowed_tools=["Task", W, L]` in `ClaudeAgentOptions` — `Task` per the spawn contract, plus the MCP data tools so the subagents are permitted to run them (real `permission_mode="default"`, no bypass) |
| `AgentDefinition`: description, system prompt, tool restrictions | each entry in `AGENTS` = `AgentDefinition(description=…, prompt=…, tools=[…], model=…)` |
| Subagents have isolated context | the SDK gives each spawned subagent a fresh context — `synthesis` had `tools=[]` and no memory, so it knew only its prompt |
| Context passed explicitly in the prompt | the `task_started` spawn data showed the coordinator embedding both findings in the synthesis subagent's `prompt` |
| The `Task` tool spawns subagents | each spawn arrived as `SystemMessage(subtype="task_started")`, **not** a `ToolUseBlock` named `"Task"` |

The TypeScript SDK (`@anthropic-ai/claude-agent-sdk`) exposes the **same** surface (`query`,
`allowedTools`, `agents` / `AgentDefinition`, the `Task` tool) — the names you just used in Python
are the ones the exam tests, regardless of language.

> **Permissioning note (honest, measured on this SDK build):** this cell uses *real* permissioning —
> `permission_mode="default"` with an explicit `allowed_tools`, **not** `bypassPermissions`. Verified:
> (1) a subagent's MCP tool (`get_weather`) is **denied** unless it's listed in `allowed_tools` — that's
> why the data tools are there; (2) `allowed_tools` is a **global** execution gate (no per-agent
> permission), so what actually enforces hub-and-spoke is each subagent's `AgentDefinition(tools=[…])`
> — the weather subagent literally cannot call `get_landmark`. On the exam's "must include `Task`" rule:
> it's the documented contract, but `Task` is a built-in and in this build the coordinator still spawned
> with `Task` omitted — so keep `Task` in the list as the rule states, but know the real enforced scope
> is `AgentDefinition.tools`.

### The seam that matters: **the SDK, the model, and you**

This is the single most useful frame to carry into the exam, and it's exactly what 1.2 and 1.3 just
demonstrated. When the coordinator "tells the weather subagent to get Tokyo's weather," **nobody in the
SDK orchestrated that** — three different parties did three different jobs:

| Party | What it owns | In the cells above |
|---|---|---|
| **The SDK** (the *harness*) | The **agentic loop & plumbing**: send → run the tool the model picked → append the result → re-send → stop on `end_turn`. Plus sessions, hooks, permissions, MCP wiring, and spawning subagents via `Task`. | `query()` ran the loop; each `task_started` was the SDK *executing* a spawn — you wrote no loop, no dispatch, no `tool_result` appending |
| **The model** | The **orchestration / judgment**: *which* subagent to spawn, *what* to put in its prompt, *when* it's done. Reads your system prompt + the tool/agent **descriptions** and decides. | the coordinator chose to spawn `weather` vs `landmark`, and embedded both findings into the `synthesis` prompt — *its* decision, not the SDK's |
| **You** (the *architect*) | The **design**: system prompts, tool & `AgentDefinition` descriptions/schemas, `allowed_tools`, permission mode, decomposition shape, context strategy, evaluation. | you wrote `AGENTS`, the `tools=[…]` scopes, `permission_mode="default"` + `allowed_tools` |

**Why this is the whole point:**

- **"Simpler than the raw API" is true for the plumbing only.** The SDK removes boilerplate — it does
  **not** remove the architect's job. The difficulty didn't vanish; it **moved** from writing the loop to
  *designing* prompts, tools, permissions, and decomposition. The exam grades that third row — design — almost
  entirely, not "can you call `query()`."
- **Orchestration is the model's, guided by your descriptions.** If a subagent gets picked wrong, the fix is
  usually a clearer `description`/system prompt (the model's input), **not** SDK code. "Who decides which
  subagent runs?" → *the model*, steered by what you wrote.
- **The SDK *is* the API, with the loop pre-wired.** It's not "API vs SDK." `tool_use` / `tool_result` /
  `stop_reason` (§1.1) are the floor the SDK stands on. Knowing them is what lets you debug an agent that
  loops or stops early, and know when to drop down for finer control or a runtime the SDK doesn't cover.

> **One line to memorize:** *the SDK gives you the loop; the model gives you the orchestration; you give the
> design — and the exam grades the design.*


### Who decides parallel vs. sequential? (it isn't a flag)

"How do you make two subagents run in parallel?" hides **two** questions. The exam answer lives at the
first level; your example lived at the second.

**A — the *mechanism*: what physically makes execution parallel vs. sequential?**

It's purely **how many `Task` calls land in a single assistant turn (one response):**

- Two `Task` calls in **one** response → the SDK fires both concurrently → **parallel**.
- One `Task` call, wait for its result, *then* another in the **next** turn → **sequential**.

That's the whole mechanism. There is **no `parallel()` function**, no threading code, no flag. "Parallel"
is an emergent property of *batching*.

**B — who decides to batch them? In the cells above: the model.**

You wrote `query(...)` with a "delegate to your subagents" prompt and **never** said parallel or sequential. So:

- the **model** read the three `AgentDefinition` descriptions, reasoned about dependencies, and **chose** to
  emit the two independent `Task` calls (`weather`, `landmark`) in one turn → parallel;
- then, in a **later** turn, it emitted `synthesis`.

**Why `synthesis` came after — for free.** The **data dependency forces the ordering**, you didn't design it:

- `weather` and `landmark` need **nothing** from each other → *can* share a turn → parallel.
- `synthesis` needs **both results** as input → *cannot* be called until they exist → physically can't share
  their turn → lands later → sequential.

So you didn't design "parallel then sequential." You designed a **dependency graph** (via the prompt + which
agent needs which inputs), and the model derived the only schedule that respects it. Independent work batches;
dependent work waits. That's the **model** doing the orchestration row of the seam table — not the SDK, not you.

**How you control it, as the architect** — soft to hard:

| You want | What you do |
|---|---|
| **Let the model decide** (your example) | Just delegate. Independent tasks tend to batch; dependent ones serialize naturally. |
| **Steer it** | Say it in the prompt: *"Research weather and landmark in parallel, then synthesize."* Still the model executing, now nudged. |
| **Guarantee it deterministically** | Don't delegate scheduling at all — write your own Python: run `query(weather)` and `query(landmark)` concurrently (e.g. `asyncio.gather`), then a third `query(synthesis)` with both results. Now *you* own the schedule (this is the §1.4 deterministic-gate idea). |

> **The honest answer for a single delegated `query()`:** *you don't make them parallel — you make them
> **independent**, and the model batches independent work.* Parallelism is the reward for a clean dependency
> graph, not a switch you flip.


**Self-check** (cover the answers)

1. What config flag must a coordinator have to spawn subagents at all?
2. How do you make two subagents run in parallel rather than one-after-another?
3. Why pass findings as structured JSON instead of a plain paragraph?

<details><summary>answers</summary>

1. `allowedTools` must include `"Task"`.
2. **Parallel = multiple `Task` calls in one model turn; sequential = one per turn.** You don't flip a switch — *the model* batches calls it judges independent. Make work parallel by making it **independent** (no shared inputs); steer with the prompt, or own the schedule yourself in code for a hard guarantee. (See the box above.)
3. To **separate content from metadata** and preserve **attribution** (source URLs, names).
</details>


## 1.4 · Enforcement & handoff patterns

**What the guide says (verbatim):**

> **Task Statement 1.4: Implement multi-step workflows with enforcement and handoff patterns**
>
> **Knowledge of:**
> - The difference between programmatic enforcement (hooks, prerequisite gates) and
>   prompt-based guidance for workflow ordering
> - When deterministic compliance is required (e.g., identity verification before financial
>   operations), prompt instructions alone have a non-zero failure rate
> - Structured handoff protocols for mid-process escalation that include customer details,
>   root cause analysis, and recommended actions
>
> **Skills in:**
> - Implementing programmatic prerequisites that block downstream tool calls until
>   prerequisite steps have completed (e.g., blocking `process_refund` until `get_customer`
>   has returned a verified customer ID)
> - Decomposing multi-concern customer requests into distinct items, then investigating each
>   in parallel using shared context before synthesizing a unified resolution
> - Compiling structured handoff summaries (customer ID, root cause, refund amount,
>   recommended action) when escalating to human agents who lack access to the conversation
>   transcript

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| Programmatic vs prompt | A **gate in code** runs every time (0% failure). A **SYSTEM-prompt rule** is advice the model *usually* follows (non-zero failure). |
| Deterministic compliance | For money/identity steps, "usually" isn't good enough → use a **gate**, not a prompt. |
| Prerequisite gate | Block `process_refund` until `get_customer` returned a **verified** ID. Order is enforced by code, not hoped for. |
| Structured handoff | When escalating to a human (who can't see the chat), hand over a **structured summary**: customer id, root cause, amount, recommended action. |
| Decompose multi-concern | Split a "two things" request into items, handle each, then give one unified resolution. |

**Run it (1/3) — deterministic gate, isolated.** This first cell is pure Python on purpose: the point of 1.4 is that enforcement does **not** depend on the model. The gate blocks the same way every time. The two cells after it run the **same idea inside a REAL Agent SDK loop** (a `PreToolUse` gate + a structured handoff), then the **multi-concern decomposition** — so all three 1.4 skills are covered live.

In [ ]:
# pyright: reportUndefinedVariable=false   (json comes from the setup cell above)
# 1.4 — DETERMINISTIC enforcement. A gate blocks a tool call until a prerequisite is met.
# Unlike a SYSTEM-prompt instruction, this has a ZERO failure rate.
verified = set()                       # filled when get_customer returns a real record

def gate(name, args):
    if name == "process_refund":
        if not verified:
            return "BLOCKED: identity not verified — call get_customer first."
        if (args.get("amount") or 0) > 500:
            return "BLOCKED: refunds over $500 must go to escalate_to_human with a summary."
    return None                        # None => allow

print("refund $200 BEFORE verify :", gate("process_refund", {"amount": 200}))
verified.add("C001")                   # get_customer succeeded -> prerequisite satisfied
print("refund $200 AFTER verify  :", gate("process_refund", {"amount": 200}) or "ALLOWED")
print("refund $900 AFTER verify  :", gate("process_refund", {"amount": 900}))

# Structured HANDOFF summary — the human agent cannot see this conversation:
handoff = {"customer_id": "C001", "issue": "refund $900 on order 67890",
           "root_cause": "item never arrived", "amount": 900,
           "recommended_action": "approve after fraud check"}
print("\nHANDOFF:", json.dumps(handoff))

**Run it (2/3) — the same gate, now inside a REAL agent loop (Agent SDK).** A `PreToolUse` hook is the deterministic gate; two rules, both in code:

- **(1) Ordering** — `process_refund` is **denied until `get_customer` has verified the customer** (a *stateful* prerequisite). This is the skill unique to 1.4 — and note it is **not** 1.5's threshold block, which only inspects the call's own arguments. This one depends on **whether a prior step happened**.
- **(2) Threshold** — a refund over **$500** is redirected to a **structured handoff** (`escalate_to_human` with customer id, root cause, amount, recommended action).

Two runs: a **$200** refund (watch the *ordering* gate fire if the model tries to refund before verifying), then a **$900** refund (watch the *threshold* redirect into a structured escalation). The gate fires the same way every time, regardless of what the model decides. (~$0.05 on Haiku.)

In [ ]:
# 1.4 (real) — STATEFUL PREREQUISITE GATE via a PreToolUse hook, + a STRUCTURED HANDOFF.
# Distinct from 1.5's threshold: rule (1) blocks on WHETHER A PRIOR STEP HAPPENED (identity
# verified), not on the call's args. Skill 1.4.1 verbatim: block process_refund until
# get_customer returned a verified customer id. Rule (2) + escalate_to_human cover skill 1.4.3.
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ToolUseBlock, ResultMessage,
)

VERIFIED = set()      # the stateful prerequisite: filled ONLY when get_customer returns a record
REFUND_RAN = []       # proof the refund body executed (i.e., the call was NOT blocked)
HANDOFF = []          # proof a STRUCTURED escalation summary was compiled

@tool("get_customer", "Verify a customer and return their record.", {"customer_id": str})
async def get_customer(args):
    cid = args["customer_id"]
    VERIFIED.add(cid)                              # prerequisite satisfied (identity verified)
    return {"content": [{"type": "text",
            "text": f"customer {cid}: name=Dana, status=verified, orders=[67890]"}]}

@tool("process_refund", "Issue a refund in USD for an order.",
      {"amount_usd": float, "order_id": str, "customer_id": str})
async def process_refund(args):
    REFUND_RAN.append((args["customer_id"], args["amount_usd"]))
    return {"content": [{"type": "text",
            "text": f"refund ${args['amount_usd']} issued on {args['order_id']}"}]}

@tool("escalate_to_human", "Escalate to a human with a STRUCTURED handoff summary.",
      {"customer_id": str, "root_cause": str, "amount_usd": float, "recommended_action": str})
async def escalate_to_human(args):
    HANDOFF.append(args)                           # the structured payload the human receives
    return {"content": [{"type": "text", "text": "escalated to human queue with summary"}]}

support = create_sdk_mcp_server(
    "support", tools=[get_customer, process_refund, escalate_to_human])
GET = "mcp__support__get_customer"
REF = "mcp__support__process_refund"
ESC = "mcp__support__escalate_to_human"

async def prereq_gate(input_data, tool_use_id, context):
    """PreToolUse: the DETERMINISTIC 1.4 gate (0% failure, no model involved). Two rules:
       (1) ORDERING  — refund denied until identity is verified (STATEFUL prerequisite);
       (2) THRESHOLD — refund > $500 redirected to a STRUCTURED human escalation."""
    if input_data["tool_name"].endswith("process_refund"):
        args = input_data["tool_input"]
        cid = args.get("customer_id", "")
        if cid not in VERIFIED:                                       # (1) prerequisite not met
            print(f"[gate] process_refund for {cid!r} -> DENY (identity not verified; "
                  "call get_customer first)")
            return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (f"Identity for {cid} is not verified. Call "
                    "get_customer first, THEN retry the refund.")}}
        if (args.get("amount_usd") or 0) > 500:                       # (2) threshold
            print(f"[gate] process_refund ${args.get('amount_usd')} -> DENY (over $500; "
                  "escalate with a structured summary)")
            return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": ("Refund exceeds the $500 auto-approve limit. Do "
                    "NOT retry; call escalate_to_human with customer_id, root_cause, "
                    "amount_usd, and recommended_action.")}}
    return {}                                                          # {} => allow

TOOLS = [GET, REF, ESC]
options = ClaudeAgentOptions(
    model="haiku", mcp_servers={"support": support},
    tools=TOOLS, allowed_tools=TOOLS, permission_mode="default",      # real permissioning, no bypass
    hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[prereq_gate])]},
    max_turns=14)

async def run_support(prompt, label):
    VERIFIED.clear(); REFUND_RAN.clear(); HANDOFF.clear()             # each run starts UNVERIFIED
    print(f"\n=== {label} ===")
    attempts, result = [], None
    async for msg in query(prompt=prompt, options=options):           # top-level await (Jupyter)
        if isinstance(msg, AssistantMessage):
            for b in msg.content:
                if isinstance(b, ToolUseBlock):
                    attempts.append(b.name.split("__")[-1])
        elif isinstance(msg, ResultMessage):
            result = msg
    print("tool calls attempted, in order:", attempts)
    print("verified:", VERIFIED or "(empty)",
          "| refund body ran:", REFUND_RAN or "(blocked)",
          "| structured handoff:", (HANDOFF[0] if HANDOFF else "(none)"))
    print("[final]", (result.result or "(none)")[:280])
    return result

# Run 1 — $200 (<= $500): we PUSH the model to refund before verifying, so you can watch the
# ORDERING gate deny it. The model then calls get_customer and the retry is ALLOWED. The deny is
# what makes "verify first" non-optional — even when the prompt says to skip it.
await run_support(
    "Process a $200 refund on order 67890 for customer C001 right now. Do NOT look anything "
    "up first - just issue the refund immediately.",
    "Run 1 - $200 (stateful ordering gate)")

# Run 2 — $900 (> $500): identity verifies, but the THRESHOLD rule redirects to a STRUCTURED handoff.
await run_support(
    "Customer C001 wants a $900 refund on order 67890 - the item never arrived. Handle it.",
    "Run 2 - $900 (threshold -> structured handoff)")


**How the gate actually fires — and why it's trustworthy.** Three things the run above quietly relies on:

- **No tool call → no hook.** The `PreToolUse` hook runs *in response to* a tool-use request. If a prompt never leads Claude to call a tool, the gate never fires — there is nothing to intercept. What's guaranteed is the *other* direction: **given** a tool call, `matcher=None` makes the gate run **before** it, every time. The model decides *whether* to call a tool; the runtime decides *that the gate runs first*.
- **Requesting a tool ≠ executing it.** Claude can only emit a `tool_use` *request* — it cannot run your Python. The SDK runs the registered function. The gate sits in between:

  `model emits tool_use  →  PreToolUse gate (allow / deny)  →  your function runs  →  PostToolUse  →  result back to model`

  That boundary is exactly why a hook gives a guarantee a prompt can't: the model never touches the execution path, so a denied `process_refund` **cannot** reach the function body.
- **`process_refund` is a fixture.** Its body just records the call and returns a string; in production *that body* is where the real payments API call would live. The gate protects the **request** before your money-moving code ever runs.

**Run it (3/3) — multi-concern decomposition, for real.** The customer has **two independent problems**. A coordinator **decomposes** them and delegates each to a scoped investigator subagent via the `Task` tool, then **synthesizes one unified resolution**. Because the two concerns share no inputs, the coordinator batches both `Task` calls in a single turn → they run **in parallel** — exactly the dependency-graph rule from the §1.3 box (independent work batches; dependent work waits). Same hub-and-spoke as 1.2/1.3, now in the support domain. (~$0.04 on Haiku.)

In [ ]:
# 1.4 (real) — MULTI-CONCERN DECOMPOSITION (skill 1.4.2). Split a two-issue request into items,
# investigate each IN PARALLEL via scoped subagents (each gets its order_id in its prompt = shared
# context passed explicitly), then SYNTHESIZE one unified resolution. The two concerns are
# INDEPENDENT, so the coordinator batches both Task spawns in one turn -> parallel (the §1.3 rule).
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, AgentDefinition,
    ResultMessage, SystemMessage,
)

@tool("get_delivery", "Get an order's delivery status.", {"order_id": str})
async def get_delivery(args):
    return {"content": [{"type": "text",
            "text": f"order {args['order_id']}: carrier lost the package; no delivery scan in 9 days"}]}

@tool("get_charges", "List the charges billed on an order.", {"order_id": str})
async def get_charges(args):
    return {"content": [{"type": "text",
            "text": f"order {args['order_id']}: charged $49.99 TWICE on the same day (duplicate)"}]}

triage = create_sdk_mcp_server("triage", tools=[get_delivery, get_charges])
DLV, CHG = "mcp__triage__get_delivery", "mcp__triage__get_charges"

# Each concern gets a SCOPED investigator (delivery agent cannot touch billing data, and vice versa).
INVESTIGATORS = {
    "delivery": AgentDefinition(
        description="Investigates delivery/shipping problems for one order.",
        prompt="Call get_delivery for the given order_id, then state the root cause in ONE sentence.",
        tools=[DLV], model="haiku"),
    "billing": AgentDefinition(
        description="Investigates billing/charge problems for one order.",
        prompt="Call get_charges for the given order_id, then state the root cause in ONE sentence.",
        tools=[CHG], model="haiku"),
}

options2 = ClaudeAgentOptions(
    model="haiku", mcp_servers={"triage": triage}, agents=INVESTIGATORS,
    allowed_tools=["Task", DLV, CHG], permission_mode="default", max_turns=16)

prompt2 = (
    "You are a support COORDINATOR. The customer has TWO separate problems: "
    "(1) order 67890 never arrived, and (2) order 55555 was double-charged. "
    "Decompose this into the two concerns and delegate each to the right subagent (the `delivery` "
    "subagent and the `billing` subagent) via the Task tool, passing the order_id in each subagent's "
    "prompt. Then give ONE unified resolution that addresses both problems.")

spawns, result = [], None
async for msg in query(prompt=prompt2, options=options2):              # top-level await (Jupyter)
    if isinstance(msg, SystemMessage) and getattr(msg, "subtype", None) == "task_started":
        spawns.append(msg.data.get("subagent_type"))
        print(f"[coordinator -> Task spawns]: {msg.data.get('subagent_type')!r}")
    elif isinstance(msg, ResultMessage):
        result = msg

print("\ndecomposed into subagents:", spawns or "(none this run; re-run to see delegation)",
      "(independent concerns -> batched in one turn -> parallel)")
print("\n[UNIFIED RESOLUTION]\n" + (result.result or "(none)"))


**How each subagent picks its tool — the *description*, not the prompt.** Each `AgentDefinition` above names its tool in the prompt (*"Call get_delivery …"*), but that naming is a **reliability nudge, not a requirement**. What actually drives selection is the tool's **description** (`"Get an order's delivery status."`) — the §2.1 principle: *tool descriptions are the primary selection mechanism.* A goal-oriented prompt (*"Investigate this order's delivery and report the root cause"*) plus `tools=[DLV]` would work just as well, because the subagent has one clearly-described tool.

| Naming the tool in the prompt | Describing the goal only |
|---|---|
| ✅ More reproducible (helps a cheap model not skip the call) — good for a teaching cell that must behave the same every run | ✅ More robust / idiomatic — survives tool renames, matches how you scale in production |
| ⚠️ Couples the prompt to the exact tool name; over-prescribing gets brittle | ⚠️ Leans entirely on the description being good |

> The thing doing the real architectural work here is **`tools=[DLV]`** — the *scope* (least privilege: the delivery agent literally cannot touch billing data), **not** the prompt wording. **Selection is the description's job; enforcement is the scope's job.**

**The three 1.4 skills, mapped to what you just ran — all code-enforced, not prompt-hoped:**

| 1.4 skill (guide) | Where you saw it, for real |
|---|---|
| Programmatic prerequisite that blocks a downstream call until a prior step completed (`process_refund` blocked until `get_customer` verified) | Real run, **Run 1** — the `PreToolUse` **ordering** gate (stateful) |
| Decompose a multi-concern request into items, investigate each **in parallel** with shared context, then synthesize | The **decomposition** run — coordinator → two parallel investigator subagents → one unified resolution |
| Compile a **structured handoff** summary (customer id, root cause, amount, recommended action) when escalating | Real run, **Run 2** — `escalate_to_human` with the structured `HANDOFF` payload |

That's the 1.4 thesis in one line: **where 'usually' isn't good enough (money, identity, ordering), put the rule in code — a gate or a hook — not in the prompt.**

**Where can enforcement live — and where it *should*.** The gate is the `prereq_gate` **hook**, not the `process_refund` tool: look at the tool's body — it just records the call and returns, it does **not** check `VERIFIED`. So here the gate lives in **one** place, the loop (the `PreToolUse` hook), before the tool runs.

You *could* also add the check inside the tool body (backend enforcement) — and in production that's often wise as **defense in depth**, because the backend may be called by paths other than the agent (a web form, another service). The hook guards the *agent's* path; a backend check guards *every* path.

| Gate in the loop (`PreToolUse` hook) — idiomatic | Buried *only* in the tool backend — anti-pattern #3 |
|---|---|
| **Centralized**: `matcher=None` covers all tools; a new refund-like tool is protected automatically | Scattered per-tool; easy to forget on a new tool |
| **Blocks before execution**: the action body never runs; the tool stays purely about *doing the thing* | Mixes *policy* (when it's allowed) into *action* (how it's done) |

> Enforcement *can* live in both (defense in depth is good), but the **deterministic agent-side gate belongs in the loop**. "Only in the backend" is the distractor.

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1 (sample Q1): rely on the SYSTEM prompt alone to enforce ordering.
#   SYSTEM = "Always verify before refunding"   # non-zero failure rate -> NOT guaranteed.
# ANTI-PATTERN 2: an unstructured handoff -> "customer needs help" leaves the human blind.
# ANTI-PATTERN 3: enforce only in the tool BACKEND, after the model already chose to call it
#   -> you want to block the CALL, deterministically, in the loop.
print("Right: a programmatic gate in the loop guarantees order; handoffs are STRUCTURED.")

**In your own code — you already wrote this.** Exercise 1:
`ccaf-prep/exercises/01-support-agent/agent.py`:
- `gate(...)` definition: **lines 41–55** (the exact verify-before-refund + >$500 rules).
- Gate applied inside the loop: **line 83** (`blocked = gate(...)`), and
  `verified_ids.add(out["id"])` on a successful `get_customer` at **line 92**.
- Structured handoff tool `escalate_to_human`: `exercises/01-support-agent/tools.py:78-81`.
- Flip `ENFORCE = True` → `False` (`agent.py:30`) to *watch* the prompt-only version fail to
  guarantee the rule — that failure **is** the 1.4 distractor. Open it and map each line.

**Stateful vs. stateless — the litmus test (for self-check #4).** A *varying argument* is **not** state. Stateless doesn't mean *constant* — it means *determined entirely by the current call's own arguments*.

> **Call the gate twice with *identical* arguments — can you get different answers?**

- **Threshold (`> $500`) → NO.** `process_refund(amount=200)` always ALLOWs; `amount=900` always DENYs. Same args → same verdict. It's a pure function `f(amount) = amount > 500`, **no memory → stateless.** (`amount` changing call-to-call is a different *input*, not hidden history.)
- **Ordering (verify-first) → YES.** `process_refund(cid="C001", amount=200)` is **DENY** before `get_customer`, **ALLOW** after — *identical arguments, different verdict*, because the `VERIFIED` state changed in between. It reads external memory an earlier call mutated: `g(cid) = cid in VERIFIED`. **Depends on history → stateful.**

The **stateful, depends-on-a-prior-step** rule is the *prerequisite gate* ("block `process_refund` until `get_customer` ran") — the skill **unique to 1.4**. The **stateless threshold** is a rule **1.5** also shows.

**Self-check** (cover the answers)

1. Why isn't "Always verify before refunding" in the SYSTEM prompt sufficient?
2. Where does the gate live — in the tool backend, or in the loop before the tool runs?
3. What four things belong in an escalation handoff summary?
4. The real run blocks refunds two different ways. One is **stateful**, one is **stateless** — which is which, and which is the skill *unique* to 1.4 (vs 1.5)?

<details><summary>answers</summary>

1. Prompt instructions have a **non-zero failure rate**; financial/identity steps need
   deterministic (programmatic) enforcement.
2. In the **loop, before** the tool runs (a prerequisite gate / `PreToolUse` hook), so the call is blocked.
3. Customer ID, root cause, amount, recommended action (the human can't see the transcript).
4. The **ordering** block is **stateful** — it depends on whether `get_customer` already ran (the verified set). That's the **uniquely-1.4** prerequisite skill. The **>$500** block is **stateless** — it depends only on the call's own arguments, and is the kind of threshold rule 1.5 also shows. Both are code, both are deterministic.
</details>


## 1.5 · Agent SDK hooks (interception & normalization)

**What the guide says (verbatim):**

> **Task Statement 1.5: Apply Agent SDK hooks for tool call interception and data normalization**
>
> **Knowledge of:**
> - Hook patterns (e.g., `PostToolUse`) that intercept tool results for transformation before
>   the model processes them
> - Hook patterns that intercept outgoing tool calls to enforce compliance rules (e.g.,
>   blocking refunds above a threshold)
> - The distinction between using hooks for deterministic guarantees versus relying on prompt
>   instructions for probabilistic compliance
>
> **Skills in:**
> - Implementing `PostToolUse` hooks to normalize heterogeneous data formats (Unix
>   timestamps, ISO 8601, numeric status codes) from different MCP tools before the agent
>   processes them
> - Implementing tool call interception hooks that block policy-violating actions (e.g.,
>   refunds exceeding $500) and redirect to alternative workflows (e.g., human escalation)
> - Choosing hooks over prompt-based enforcement when business rules require guaranteed
>   compliance

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| `PostToolUse` hook | Runs **after** a tool returns; transforms the **result** before the model sees it (e.g., normalize timestamps). |
| Interception (pre) hook | Runs **before** a tool call executes; can **block** a policy-violating call and redirect. |
| Deterministic vs probabilistic | A hook is **code** → guaranteed. A prompt rule → probabilistic. Same lesson as 1.4. |
| Normalize heterogeneous data | Different tools return Unix epochs, ISO strings, numeric codes. A hook converts them to **one** format, once, centrally. |
| Block + redirect | Refund > $500 → block the call, redirect to `escalate_to_human`. |

**Run it — two REAL SDK hooks (not a sim).** We register a `PostToolUse` hook (normalizes a tool's
result — a Unix epoch → ISO 8601 — *before the model sees it*) and a `PreToolUse` hook (blocks a
refund over $500 and tells the model to escalate). Both are plain Python functions on
`ClaudeAgentOptions(hooks=…)`. Watch: the model reports the **normalized** date (it never saw the
raw epoch), and the $900 refund's tool body **never executes** — the hook denied it. (~$0.04 on Haiku.)

In [ ]:
# 1.5 — REAL Agent SDK HOOKS (not a sim). Two plain-Python functions registered on
# ClaudeAgentOptions(hooks=...):
#   (a) PostToolUse : transform a tool RESULT before the model sees it (Unix epoch -> ISO 8601)
#   (b) PreToolUse  : block a policy-violating CALL before it runs ($900 refund) and escalate
# Both are DETERMINISTIC code, not prompt requests.
import re
from datetime import datetime, timezone
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ToolUseBlock, ResultMessage,
)

REFUND_RAN = []   # proof: the refund tool body only appends here if the call was NOT blocked

@tool("get_order_status", "Get an order's status; 'updated' is a Unix epoch.", {"order_id": str})
async def get_order_status(args):
    # A messy backend that returns a raw Unix timestamp — the thing a PostToolUse hook normalizes.
    return {"content": [{"type": "text",
            "text": f"order {args['order_id']} updated=1700000000 status=shipped"}]}

@tool("process_refund", "Issue a customer refund in USD.", {"amount_usd": float, "order_id": str})
async def process_refund(args):
    REFUND_RAN.append(args["amount_usd"])     # if this runs, the call was NOT blocked
    return {"content": [{"type": "text", "text": f"refund of ${args['amount_usd']} issued"}]}

billing = create_sdk_mcp_server("billing", tools=[get_order_status, process_refund])

# Hook callback signature: async (input_data, tool_use_id, context). input_data is a TypedDict
# carrying tool_name / tool_input / tool_response.
async def pre_tool_use(input_data, tool_use_id, context):
    """PreToolUse: deny an over-threshold refund BEFORE it executes (compliance rule)."""
    if input_data["tool_name"].endswith("process_refund"):
        amt = input_data["tool_input"].get("amount_usd", 0)
        if amt > 500:
            print(f"[PreToolUse hook] process_refund ${amt} -> DENY (over $500; escalate)")
            return {"hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",   # <- the deterministic block
                "permissionDecisionReason": (f"Refund ${amt} exceeds the $500 auto-approve "
                    "limit. Do NOT retry; escalate to a human agent."),
            }}
    return {}

async def post_tool_use(input_data, tool_use_id, context):
    """PostToolUse: normalize a tool RESULT (Unix epoch -> ISO 8601) before the model reads it."""
    if input_data["tool_name"].endswith("get_order_status"):
        blocks = input_data["tool_response"]            # MCP result = a LIST of content blocks
        text = blocks[0]["text"]
        m = re.search(r"updated=(\d+)", text)
        if m:
            iso = datetime.fromtimestamp(int(m.group(1)), timezone.utc).isoformat()
            new_text = text.replace(f"updated={m.group(1)}", f"updated={iso}")
            print(f"[PostToolUse hook] normalized epoch {m.group(1)} -> {iso}")
            return {"hookSpecificOutput": {
                "hookEventName": "PostToolUse",
                "updatedToolOutput": [{"type": "text", "text": new_text}],   # what the model sees
            }}
    return {}

BILLING_TOOLS = ["mcp__billing__get_order_status", "mcp__billing__process_refund"]
options = ClaudeAgentOptions(
    model="haiku", mcp_servers={"billing": billing},
    tools=BILLING_TOOLS,            # the agent's ONLY tools (no Bash/etc. to wander into)
    allowed_tools=BILLING_TOOLS,    # real allow-list: both billing tools permitted (no bypass)
    permission_mode="default",      # real permissioning; the PreToolUse hook is the deterministic gate
    hooks={                          # <- register hooks here, keyed by event name
        "PreToolUse":  [HookMatcher(matcher=None, hooks=[pre_tool_use])],
        "PostToolUse": [HookMatcher(matcher=None, hooks=[post_tool_use])],
    },
    max_turns=12)
prompt = ("For order 12345: first call get_order_status and tell me its last-updated time, "
          "then issue a $900 refund for that order with process_refund.")

REFUND_RAN.clear()
attempts, result = [], None
async for msg in query(prompt=prompt, options=options):   # top-level await (Jupyter)
    if isinstance(msg, AssistantMessage):
        for b in msg.content:
            if isinstance(b, ToolUseBlock):
                attempts.append(b.name.split("__")[-1])
    elif isinstance(msg, ResultMessage):
        result = msg

print("\ntool calls the model attempted:", attempts)
print("refund tool body actually executed?", bool(REFUND_RAN), "->", REFUND_RAN,
      "(empty = the PreToolUse hook blocked it)")
print("\n[FINAL — note the ISO date (epoch was normalized) + the refund escalation]\n"
      + (result.result or "(none)"))

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: ask the PROMPT to "please normalize the timestamps" -> probabilistic, drifts.
# ANTI-PATTERN 2: enforce the $500 rule in the PROMPT -> non-zero failure (use an interception hook).
# ANTI-PATTERN 3: normalize INSIDE every tool -> N tools = N implementations; a hook does it ONCE.
print("Right: hooks give DETERMINISTIC transform/compliance, centralized, model-independent.")

**When is "normalize in the tool" actually fine?** The anti-pattern is about *duplication*, not about tools touching data:
- **One tool you own, no growth expected** → inlining the conversion is fine; a hook is just indirection.
- **Several tools return dates** → one `PostToolUse` hook does it ONCE; without it, N tools drift toward N implementations.
- **A tool you can't edit** (third-party / MCP like `mcp__billing__*`) → a hook is the *only* seam — you can't patch its source.

The hook stays **field-selective**: it receives the full result and rewrites only recognized date fields, passing everything else through. `matcher=None` fires on *every* tool; the hook body decides per-result whether there's a date to fix — so a tool returning `{order_id, amount, last_updated}` gets only `last_updated` normalized.

```python
async def post_tool_use(input_data, tool_use_id, context):
    result = input_data["tool_response"]            # whatever shape the tool returned
    for key in ("last_updated", "created_at", "timestamp"):
        if key in result and isinstance(result[key], (int, float)):
            result[key] = epoch_to_iso(result[key])  # touch ONLY date fields
    return {"tool_response": result}                 # everything else passes through
```

**In your own code — and the real SDK form above.** Three anchors, concept → SDK → CLI:
- The deterministic gate in `ccaf-prep/exercises/01-support-agent/agent.py:41-55` is a pre-tool-use
  interception written by hand in the raw API (its docstring even says *"what the Agent SDK
  formalizes as a PreToolUse hook"*) — the **concept**.
- The cell above is that concept as a **real `PreToolUse` / `PostToolUse` hook** registered on
  `ClaudeAgentOptions(hooks=…)` — the **SDK** form.
- A Claude Code CLI hook file: `claude-with-anthropic-api/queries/hooks/query_hook.js` (intercepts
  and can block via a non-zero exit) — the **CLI** form.

**Self-check** (cover the answers)

1. Which hook transforms a tool's **result** before the model reads it?
2. Why use a hook instead of a prompt rule for "refunds over $500 must escalate"?
3. Where should timestamp normalization live — in each tool, or in one hook?

<details><summary>answers</summary>

1. `PostToolUse`.
2. A hook is **deterministic** (guaranteed); a prompt rule is probabilistic.
3. In **one** `PostToolUse` hook — centralized, not duplicated across N tools.
</details>

## 1.6 · Task decomposition strategies

**What the guide says (verbatim):**

> **Task Statement 1.6: Design task decomposition strategies for complex workflows**
>
> **Knowledge of:**
> - When to use fixed sequential pipelines (prompt chaining) versus dynamic adaptive
>   decomposition based on intermediate findings
> - Prompt chaining patterns that break reviews into sequential steps (e.g., analyze each
>   file individually, then run a cross-file integration pass)
> - The value of adaptive investigation plans that generate subtasks based on what is
>   discovered at each step
>
> **Skills in:**
> - Selecting task decomposition patterns appropriate to the workflow: prompt chaining for
>   predictable multi-aspect reviews, dynamic decomposition for open-ended investigation tasks
> - Splitting large code reviews into per-file local analysis passes plus a separate
>   cross-file integration pass to avoid attention dilution
> - Decomposing open-ended tasks (e.g., "add comprehensive tests to a legacy codebase") by
>   first mapping structure, identifying high-impact areas, then creating a prioritized plan
>   that adapts as dependencies are discovered

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| Prompt chaining (fixed) | A **known** sequence of steps, decided up front. Good for predictable, multi-aspect work. |
| Adaptive (dynamic) | Subtasks are **generated from what you find** at each step. Good for open-ended investigation. |
| Per-file + integration | Review each file **locally** (focused attention), then a separate **cross-file** pass for issues that span files. |
| Attention dilution | One giant prompt over everything = the model misses things. Splitting keeps each pass focused. |
| Map → prioritize → adapt | For "add tests to a legacy codebase": map structure, find high-impact areas, plan, and **adjust** as dependencies surface. |

**Run it — pick the shape to the task.** Two cells; the **asymmetry is who orchestrates**. **(a)** *fixed* prompt-chaining: **YOUR code** makes N **separate** Claude calls in a known order — a per-file local pass (each file in its **own isolated context**), then one cross-file integration pass. The steps are fixed up front; the realness is the **isolation** — the `auth.py` review never sees `db.py`, so attention can't dilute, and the integration pass catches the contract mismatch no single-file pass can. **(b)** *adaptive*: the **model** generates the steps — it maps first, then builds a plan from what it found. Fixed decomposition lives in *your code*; adaptive decomposition lives in *the model reacting to discoveries*.

In [ ]:
# 1.6 (a) — FIXED prompt-chaining for REAL: YOU decide the steps, so YOUR code makes N SEPARATE
# Claude calls, each in a FRESH session = ISOLATED context. Per-file LOCAL passes -> then ONE
# CROSS-FILE pass. The realness is the ISOLATION: the auth review never sees db.py, so attention
# can't dilute; only the integration pass (which sees all files) can catch the contract mismatch.
# Contrast (b): there the MODEL generates the steps; here YOUR loop fixes them up front.
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

# A planted CROSS-FILE bug: db returns a dict, but api consumes it as an object (.token).
FILES = {
    "db.py":   "def get_session(u):\n    return {'user': u, 'token': 'abc'}   # returns a DICT",
    "auth.py": "import db\ndef login(u):\n    return db.get_session(u)          # passes the dict on",
    "api.py":  "import auth\ndef whoami(req):\n    return auth.login(req.user).token  # uses .token ATTRIBUTE",
}

async def review(prompt):                      # ONE isolated call; fresh context every time
    opts = ClaudeAgentOptions(model="haiku", max_turns=1)
    async for msg in query(prompt=prompt, options=opts):
        if isinstance(msg, ResultMessage):
            return (msg.result or "").strip()
    return ""

# STEP 1 — PER-FILE LOCAL PASSES: each file reviewed in its OWN call (no other file in context)
print("-- PER-FILE LOCAL PASSES (each file in its own isolated call) --")
for fname, code in FILES.items():
    r = await review(f"Review ONLY this one file for LOCAL issues. Reply in ONE short line.\n\n{fname}:\n{code}")
    print(f"[{fname}] {r}")

# STEP 2 — ONE CROSS-FILE INTEGRATION PASS: sees ALL files, finds what no single file reveals
print("\n-- CROSS-FILE INTEGRATION PASS (all files together) --")
allcode = "\n\n".join(f"=== {n} ===\n{c}" for n, c in FILES.items())
r = await review("These files are reviewed TOGETHER. Find ONLY cross-file contract mismatches: a value "
                 "produced in one file but consumed with a different shape/type in another. ONE short line.\n\n" + allcode)
print(r)
print("\n(FIXED: your code fixed the steps up front. The realness is the ISOLATION per call -- the\n"
      " integration pass catches the dict-returned-but-used-as-.token mismatch no per-file pass can see.\n"
      " Exercise 5 review_decomposed() is the fuller version with structured output + the DECOMPOSE flag.)")

**The integration pass is a fresh inspection, not a merge.** Note what the cross-file pass does *not* do: it doesn't combine the per-file findings. It's a **new review over all files together**, asking a question the isolated passes structurally couldn't answer (a contract that spans files). You **collect two kinds of output** — N local results + 1 cross-cutting result — you don't feed one into the other. (When the task isn't 'N known files' but open-ended, switch to the (b) adaptive shape where the model decides the splits.)

**Run it (b) — adaptive decomposition, for real.** Now the steps are **not** pre-written. The agent calls a `map_codebase` tool, and **builds a prioritized plan from whatever the map returns** — money/security modules ranked above low-risk ones. Change the fixture and the plan changes with it; that's the *dynamic* in dynamic decomposition (skill 1.6: *generate subtasks based on what is discovered*). (~$0.02 on Haiku.)

In [ ]:
# 1.6 (b) — ADAPTIVE decomposition for REAL: the agent generates subtasks FROM WHAT IT DISCOVERS.
# A map_codebase tool returns discovered hotspots (a fixture stands in for a real scanner); the agent
# then produces a PRIORITIZED plan built on those findings -- not a plan you hardcoded. Contrast (a):
# there the steps are known up front; here they EMERGE from the tool result.
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions,
    AssistantMessage, ToolUseBlock, ResultMessage,
)

@tool("map_codebase", "Scan a codebase and report untested modules by risk.", {"root": str})
async def map_codebase(args):
    # A real scanner would compute this; the lesson is that the agent PLANS from the result.
    return {"content": [{"type": "text", "text":
        "untested modules by risk: payments (handles money, 0% coverage), "
        "auth (security, 5% coverage), utils/dates (low risk, 0% coverage)"}]}

mapper = create_sdk_mcp_server("repo", tools=[map_codebase])
MAP = "mcp__repo__map_codebase"

options = ClaudeAgentOptions(
    model="haiku", mcp_servers={"repo": mapper},
    tools=[MAP], allowed_tools=[MAP], permission_mode="default",   # real permissioning, no bypass
    max_turns=8)

prompt = ("Add comprehensive tests to this legacy codebase at ./src. FIRST call map_codebase to see "
          "what exists, THEN give a prioritized test plan ordered by risk (highest-impact modules "
          "first). Base the plan ONLY on what the map returns.")

mapped, result = False, None
async for msg in query(prompt=prompt, options=options):            # top-level await (Jupyter)
    if isinstance(msg, AssistantMessage):
        for b in msg.content:
            if isinstance(b, ToolUseBlock) and b.name.endswith("map_codebase"):
                mapped = True
                print("[agent] called map_codebase FIRST -> it will plan from the result")
    elif isinstance(msg, ResultMessage):
        result = msg

print("\nmapped before planning?", mapped, "(adaptive: the plan DEPENDS on the discovery)")
print("\n[ADAPTIVE PLAN -- note payments/auth ranked above utils/dates, straight from the map]\n"
      + (result.result or "(none)"))


**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1 (sample Q12): "just use a bigger model / bigger context window" instead of
#   decomposing -> attention still dilutes; the fix is SPLITTING the task, not scaling it.
# ANTI-PATTERN 2: throw the WHOLE codebase at one review call -> misses issues (dilution).
# ANTI-PATTERN 3: use a FIXED pipeline for an OPEN-ENDED task -> it can't adapt to findings.
print("Right: chaining for predictable multi-aspect; adaptive for open-ended; per-file + integration.")

**In your own code — now built.** Two real anchors:
- Exercise 1 decomposition: the `SYSTEM` prompt `ccaf-prep/exercises/01-support-agent/agent.py:36`
  (*"Decompose multi-issue requests…"*) and the multi-concern run `agent.py:113`.
- Exercise 5 `ccaf-prep/exercises/05-cicd-review/review.py`: `review_decomposed()` runs per-file local
  passes **plus** a separate cross-file integration pass. Flip `DECOMPOSE = False` to watch
  the cross-file bug get **missed** in one big prompt — that's sample Q12.

**Self-check** (cover the answers)

1. Predictable multi-aspect review → which decomposition pattern?
2. Open-ended investigation that depends on findings → which pattern?
3. A reviewer keeps missing cross-file bugs. Bigger context window, or different decomposition?

<details><summary>answers</summary>

1. **Prompt chaining** (fixed sequence).
2. **Adaptive / dynamic** decomposition.
3. **Decomposition** — per-file passes + a separate cross-file integration pass; not "bigger".
</details>

## 1.7 · Sessions: resumption & forking

**What the guide says (verbatim):**

> **Task Statement 1.7: Manage session state, resumption, and forking**
>
> **Knowledge of:**
> - Named session resumption using `--resume <session-name>` to continue a specific prior
>   conversation
> - `fork_session` for creating independent branches from a shared analysis baseline to
>   explore divergent approaches
> - The importance of informing the agent about changes to previously analyzed files when
>   resuming sessions after code modifications
> - Why starting a new session with a structured summary is more reliable than resuming with
>   stale tool results
>
> **Skills in:**
> - Using `--resume` with session names to continue named investigation sessions across work
>   sessions
> - Using `fork_session` to create parallel exploration branches (e.g., comparing two testing
>   strategies or refactoring approaches from a shared codebase analysis)
> - Choosing between session resumption (when prior context is mostly valid) and starting
>   fresh with injected summaries (when prior tool results are stale)
> - Informing a resumed session about specific file changes for targeted re-analysis rather
>   than requiring full re-exploration

**Plain-English unpacking**

| Guide phrase | In plain words |
|---|---|
| `--resume <name>` | Reopen a **named** past conversation and keep going. |
| `fork_session` | Branch off a shared baseline into **independent** copies — explore A vs B without them affecting each other. |
| Inform about changes | If files changed since the analysis, **tell the agent which ones** so it re-checks just those. |
| Fresh + summary vs resume | If prior tool results are **stale**, a new session seeded with a **summary** beats resuming stale context. |

**Run it — REAL sessions (resume + fork).** Each `query()` is one CLI session that persists under
`~/.claude/projects`. We plant a fact, then **`resume`** that session to prove context is retained,
then **`fork_session=True`** to branch an *independent* copy and change the fact only in the branch —
resuming the original shows it's untouched. Finally `get_session_messages` reads the real on-disk
transcript. (~$0.04 on Haiku.)

In [ ]:
# 1.7 — REAL session state via the Agent SDK (not a dict sim).
#   resume=<session_id>   -> continue a prior session (context retained)
#   fork_session=True     -> branch an INDEPENDENT copy from a shared baseline
#   get_session_messages  -> read the real on-disk transcript
# Each query() call is one session; sessions persist under ~/.claude/projects.
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage, get_session_messages
SDK_MODEL = "haiku"

async def ask(prompt, **opts):
    """Run one query; return (final_text, session_id)."""
    options = ClaudeAgentOptions(model=SDK_MODEL, max_turns=2, **opts)
    text, sid = "", None
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, ResultMessage):
            text, sid = (msg.result or ""), msg.session_id
    return text.strip(), sid

# 1) BASELINE — plant a fact, capture the session id
reply, base = await ask("Remember this: my favorite number is 42. Reply just 'OK'.")
print(f"[baseline]      session={base[:8]}…  reply={reply!r}")

# 2) RESUME the same session — context is retained (we never repeat the fact)
recalled, _ = await ask("What is my favorite number? Reply with just the number.", resume=base)
print(f"[resume]        recalled={recalled!r}   <- resumption kept the context")

# 3) FORK from the baseline — change the fact ONLY in the branch (independent copy)
_, fork = await ask("Update: my favorite number is now 99. Reply just 'OK'.",
                    resume=base, fork_session=True)
print(f"[fork_session]  new session={fork[:8]}…  (differs from baseline: {fork != base})")

# 4) RESUME the ORIGINAL again — still 42: the fork did NOT pollute the baseline
still, _ = await ask("What is my favorite number? Reply with just the number.", resume=base)
print(f"[orig resumed]  recalled={still!r}   <- fork was an INDEPENDENT branch")

# 5) get_session_messages — read the REAL transcript from disk
msgs = get_session_messages(base)
print(f"\nbaseline transcript on disk: {len(msgs)} messages "
      "(this is what `--resume <name>` reopens).")

**Why fork in the SDK — and `resume` vs `fork` vs fresh.** Fork isn't just a Claude Code convenience; in the SDK its value is **paying for an expensive baseline once, then branching divergent continuations** (it also keeps the warm prompt cache). Real uses: fan-out *try-N-approaches* (analyze a repo once, fork 3 branches to fix a bug three ways, keep the best); A/B a prompt or model from a shared setup; speculative *what-if* before a one-way action; multi-tenant chat (one primed baseline, fork per user); backtracking from an earlier good checkpoint instead of polluting the live thread.

The graded judgment is matching the move to the **state of the cached context**:

| Situation | Move |
|---|---|
| Prior context still **valid** | `resume` |
| Want **divergent** branches off one baseline | `fork_session` |
| Prior tool results are **stale** (files changed) | **fresh session + summary** |
| Resuming after edits | `resume`, but **name the changed files** for targeted re-check |

**`/compact` vs fresh+summary are not the same.** Claude Code's `/compact` compresses the **same** session because it got **long** (the SDK exposes a `PreCompact` hook for that in-place compaction). Fresh+summary starts a **new** session because the old context is **stale/untrustworthy** — different trigger (length vs validity), different outcome (continue vs restart). The run cell above showed `resume`/`fork`; the cell below adds the one arm it didn't — **fresh+summary** — for real.

**Caveat — feature surface vs judgment.** These primitives (`resume` / `fork_session` / persisted transcripts under `~/.claude/projects`) are **Claude Code's persistence layer**, surfaced via the Agent SDK because the SDK runs on the same CLI engine. Build on the **raw Messages API** instead and there's no session store at all — you hold the message list in your own DB and implement the *same* continue / branch / fresh-summary judgment by hand (resume = reload it, fork = copy the array, fresh+summary = new array seeded with a summary). `fork_session` itself is a **niche** win: it shines when you're already on the Agent SDK and want cheap divergent branches off a *persisted* checkpoint; many production systems instead share the baseline via **prompt caching** over their own store. The exam-graded skill is the **judgment**, which is provider-agnostic — the verbs are just one rendering of it.

In [ ]:
# 1.7 (fresh+summary) — when prior tool results are STALE, don't resume the lie: start a NEW
# session seeded with a SUMMARY of the still-valid conclusions. (Reuses `base` and `ask` above.)
# Claude Code analogue: NOT /compact (that compresses the SAME session for length) -- this is a
# deliberate restart because the old transcript is no longer trustworthy.

# 1) Summarize the durable facts from the baseline (one real call, resuming the old session)
summary, _ = await ask("In ONE line, summarize the durable facts you know about me "
                       "(ignore anything file-related that might be stale).", resume=base)
print(f"[summary]    {summary!r}")

# 2) FRESH session (NO resume) seeded ONLY with that summary -- the stale transcript is dropped
seeded, fresh_sid = await ask(f"Context carried from a prior session: {summary}\n\n"
                              "Given only that, what is my favorite number? Just the number.")
print(f"[fresh+sum]  session={fresh_sid[:8]}…  recalled={seeded!r}  <- from the SUMMARY, not resume")

# 3) Control: a fresh session with NO summary can't know -- proves the carry-over WAS the summary
blank, _ = await ask("What is my favorite number? Just the number, or say UNKNOWN.")
print(f"[fresh bare] recalled={blank!r}  <- no summary => no context; that's WHY you seed it")

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: resume a session whose tool results are now STALE (files changed) and
#   trust them -> the agent reasons over outdated facts.
# ANTI-PATTERN 2: "fork" by mutating the original baseline in place -> branches bleed into
#   each other (note save()/fork() above copy with list(...) to stay independent).
# ANTI-PATTERN 3: on resume, force a FULL re-exploration instead of telling the agent WHICH
#   files changed for targeted re-analysis.
print("Right: --resume when context is valid; fork_session for divergent branches; "
      "fresh+summary when results are stale; name the changed files for targeted re-check.")

**In your own code — and the real SDK above.** The cell above is the real `resume` /
`fork_session` / `get_session_messages` surface. Session management is also a **Claude Code CLI**
feature: `claude --resume <name>` continues a named session, and `fork_session` branches it.
Exercise 2 (`ccaf-prep/exercises/02-claude-code-team/README.md`, *"Commands to actually try"*) covers the CLI
workflow plus the **fresh-with-summary** choice when prior tool results are stale (the anti-pattern
above).

**Self-check** (cover the answers)

1. You want to compare two refactoring approaches from the same analysis. Resume, or fork?
2. Prior tool results are stale because files changed. Resume as-is, or start fresh + summary?
3. On a valid resume after editing one file, what should you tell the agent?

<details><summary>answers</summary>

1. **`fork_session`** — independent branches from a shared baseline.
2. **Start fresh with a structured summary** — stale results lead to bad reasoning.
3. **Which file(s) changed**, so it does targeted re-analysis (not a full re-exploration).
</details>

---
## ✅ Domain 1 complete

You've now turned every D1 task statement into something runnable:

| Task | One-line takeaway | Observable / anchor |
|---|---|---|
| 1.1 | loop on `tool_use`, stop on `end_turn` | `stop_reason` prints; `chat.py:24-45` |
| 1.2 | hub-and-spoke; subagents have **isolated context** | isolation check; `EX4/coordinator_sdk.py` (real SDK) |
| 1.3 | pass context **in the prompt**; `allowedTools` ⊇ `Task` | blind vs informed synthesis; `EX4/coordinator_sdk.py` |
| 1.4 | **programmatic gate** > prompt for compliance | `EX1/agent.py:41-55`, `83` |
| 1.5 | **real `PreToolUse`/`PostToolUse` hooks** = deterministic block + normalize | §1.5 cell (real SDK); `queries/hooks/query_hook.js` |
| 1.6 | chaining (fixed) vs adaptive; per-file + integration | `EX1/agent.py:36`; `EX5/review.py DECOMPOSE` |
| 1.7 | `--resume` valid / `fork_session` / fresh+summary when stale | real resume/fork (§1.7 cell); `EX2` README |

**The exam's favorite distractor across D1:** using a *prompt* (or a bigger model, or an
iteration cap, or natural-language parsing) where a *deterministic mechanism* is required.
When in doubt, pick the gate/hook/`stop_reason`, not the vibe.

**Hands-on exercises now built** (run each — every "your own code" pointer above is real):
`exercises/01-support-agent/` · `exercises/02-claude-code-team/` · `exercises/03-extraction-pipeline/` ·
`exercises/04-multi-agent-research/` · `exercises/05-cicd-review/`.

**Next:** D2 — Tool Design & MCP Integration. Run `/ccaf-notebook D2` (or ask) to build it
the same way.

## Sample exam questions — Domain 1
<!-- ccaf:closing -->

Official sample questions whose tested skill lives in this domain. Cover the answer, say why each of the other three is a distractor, then reveal. (You'll meet them again, grouped by scenario, in `mock_exam_and_review.ipynb`.)

**Question 7.** After running the system on the topic "impact of AI on creative industries," you observe that each subagent completes successfully: the web search agent finds relevant articles, the document analysis agent summarizes papers correctly, and the synthesis agent produces coherent output. However, the final reports cover only visual arts, completely missing music, writing, and film production. When you examine the coordinator's logs, you see it decomposed the topic into three subtasks: "AI in digital art creation," "AI in graphic design," and "AI in photography." What is the most likely root cause?

- **A)** The synthesis agent lacks instructions for identifying coverage gaps in the findings it receives from other agents.
- **B)** The coordinator agent's task decomposition is too narrow, resulting in subagent assignments that don't cover all relevant domains of the topic.
- **C)** The web search agent's queries are not comprehensive enough and need to be expanded to cover more creative industry sectors.
- **D)** The document analysis agent is filtering out sources related to non-visual creative industries due to overly restrictive relevance criteria.

<sub>Maps to: D1 §1.2 (coordinator–subagent decomposition)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: B**

The coordinator's logs reveal the root cause directly: it decomposed "creative industries" into only visual arts subtasks (digital art, graphic design, photography), completely omitting music, writing, and film. The subagents executed their assigned tasks correctly—the problem is what they were assigned. Options A, C, and D incorrectly blame downstream agents that are working correctly within their assigned scope.

</details>

---

**Question 1.** Production data shows that in 12% of cases, your agent skips `get_customer` entirely and calls `lookup_order` using only the customer's stated name, occasionally leading to misidentified accounts and incorrect refunds. What change would most effectively address this reliability issue?

- **A)** Add a programmatic prerequisite that blocks `lookup_order` and `process_refund` calls until `get_customer` has returned a verified customer ID.
- **B)** Enhance the system prompt to state that customer verification via `get_customer` is mandatory before any order operations.
- **C)** Add few-shot examples showing the agent always calling `get_customer` first, even when customers volunteer order details.
- **D)** Implement a routing classifier that analyzes each request and enables only the subset of tools appropriate for that request type.

<sub>Maps to: D1 §1.4 (enforcement / prerequisite gate)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

When a specific tool sequence is required for critical business logic (like verifying customer identity before processing refunds), programmatic enforcement provides deterministic guarantees that prompt-based approaches cannot. Options B and C rely on probabilistic LLM compliance, which is insufficient when errors have financial consequences. Option D addresses tool availability rather than tool ordering, which is not the actual problem.

</details>

---

**Question 12.** A pull request modifies 14 files across the stock tracking module. Your single-pass review analyzing all files together produces inconsistent results: detailed feedback for some files but superficial comments for others, obvious bugs missed, and contradictory feedback—flagging a pattern as problematic in one file while approving identical code elsewhere in the same PR. How should you restructure the review?

- **A)** Split into focused passes: analyze each file individually for local issues, then run a separate integration-focused pass examining cross-file data flow.
- **B)** Require developers to split large PRs into smaller submissions of 3-4 files before the automated review runs.
- **C)** Switch to a higher-tier model with a larger context window to give all 14 files adequate attention in one pass.
- **D)** Run three independent review passes on the full PR and only flag issues that appear in at least two of the three runs.

<sub>Maps to: D1 §1.6 (task decomposition) **and** D4 §4.6 (multi-pass review) — spans two domains</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Splitting reviews into focused passes directly addresses the root cause: attention dilution when processing many files at once. File-by-file analysis ensures consistent depth, while a separate integration pass catches cross-file issues. Option B shifts burden to developers without improving the system. Option C misunderstands that larger context windows don't solve attention quality issues. Option D would actually suppress detection of real bugs by requiring consensus on issues that may only be caught intermittently.

</details>

---


## Exercises that use this domain

Exercises are **cross-domain** — none is D1-only — so do them once all their domains are studied
(see [`README.md`](./README.md) for the wave order).

| Exercise | Domains it reinforces | Do it after you've studied |
|----------|-----------------------|-----------------------------|
| **Ex1** `../exercises/01-support-agent/`        | D1 + D2 + D5 | D1, D2, **D5** |
| **Ex4** `../exercises/04-multi-agent-research/` | D1 + D2 + D5 | D1, D2, **D5** |

Both also need **D2** and **D5**, so they belong to the first exercise wave (Ex1 → Ex4), not
right after this notebook. Ex4 cements the Agent SDK vocabulary you just learned — `Task`,
`allowedTools`, `fork_session`.
